In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col , count , round, when

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 3, Finished, Available, Finished, False)

In [5]:
df.printSchema()
df.show(truncate=False)

StatementMeta(, 422f9520-ec9c-48fe-994a-0cb4559f32e2, 7, Finished, Available, Finished, False)

root
 |-- Year: string (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- DayofMonth: string (nullable = true)
 |-- DayOfWeek: string (nullable = true)
 |-- FlightDate: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: string (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- OriginAirportID: string (nullable = true)
 |-- OriginAirportSeqID: string (nullable = true)
 |-- OriginCityMarketID: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: string (nullable = true)
 |-- DestAirportID: string (nullable = true)
 |-- DestAirportSeqID: string (nul

In [1]:
df = spark.read.csv(
    "Files/bronze/raw_extracted/",
    header=True,
    inferSchema=False
)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 3, Finished, Available, Finished, False)

In [4]:
selected_df = df.select(
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "OriginAirportID",
    "DestAirportID",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
)



StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 6, Finished, Available, Finished, False)

In [5]:
selected_df.printSchema()
display(selected_df)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 7, Finished, Available, Finished, False)

root
 |-- FlightDate: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- OriginAirportID: string (nullable = true)
 |-- DestAirportID: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- Cancelled: string (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- CarrierDelay: string (nullable = true)
 |-- WeatherDelay: string (nullable = true)
 |-- NASDelay: string (nullable = true)
 |-- SecurityDelay: string (nullable = true)
 |-- LateAircraftDelay: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 8fac3104-dc1b-4294-bad7-1cfe89a92268)

In [6]:
total_rows = selected_df.count()

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 8, Finished, Available, Finished, False)

**Creating a function to show null percentages**

In [7]:
def null_percentage(df):
    total_rows = df.count()

    null_percentages = df.select(
        *[
            round(
                count(when(col(c).isNull(), c)) / total_rows * 100,
                2
            ).alias(c)
            for c in df.columns
        ]
    )

    return null_percentages

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 9, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import count, when, col, round

display(null_percentage(selected_df))

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cf804970-9cc9-40e5-8678-b37043a98527)

In [13]:
# casting first
selected_df = (selected_df
    .withColumn("Cancelled", col("Cancelled").cast("double").cast("boolean"))
    .withColumn("ArrDelay", col("ArrDelay").cast("double"))
    .withColumn("DepDelay", col("DepDelay").cast("double"))
)
bad_rows = selected_df.filter(
    (col("ArrDelay").isNull()) &
    (col("Cancelled") == False) &
    (col("Diverted") == False)
)


StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 15, Finished, Available, Finished, False)

In [14]:
selected_df.printSchema()
display(selected_df)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 16, Finished, Available, Finished, False)

root
 |-- FlightDate: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- OriginAirportID: string (nullable = true)
 |-- DestAirportID: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- Cancelled: boolean (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- CarrierDelay: string (nullable = true)
 |-- WeatherDelay: string (nullable = true)
 |-- NASDelay: string (nullable = true)
 |-- SecurityDelay: string (nullable = true)
 |-- LateAircraftDelay: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 004712a8-6c7a-4273-bc31-ffa6515abbe8)

In [15]:
departure_bad_rows = selected_df.filter(
    (col("DepDelay").isNull()) &
    (col("Cancelled") == False) &
    (col("Diverted") == False)
)

departure_bad_rows.count()

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 17, Finished, Available, Finished, False)

0

In [16]:
display(selected_df)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e4351f4-9b42-46f3-ac66-425c6cc95adc)

In [17]:
bad_delay_cause = df.filter(
    (col("CarrierDelay").isNull()) &
    (col("ArrDelay").isNotNull()) &
    (col("ArrDelay") >= 15)
)
bad_delay_cause.count()

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 19, Finished, Available, Finished, False)

0

### Dropping meaningless nulls (Cancelled and Diverted)

In [18]:
display(selected_df.filter(col("Cancelled").isNull() | col("Diverted").isNull() | col("Distance").isNull()))

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c38243d2-8f90-4037-a773-32bb94c09df3)

In [19]:
selected_df = selected_df.filter(col("Cancelled").isNotNull() & col("Diverted").isNotNull() & col("Distance").isNotNull())

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 21, Finished, Available, Finished, False)

In [20]:
display(null_percentage(selected_df))

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc65588f-d5c0-4462-a9fe-eac42de1775c)

##### Casting

In [21]:
from pyspark.sql.functions import to_date

selected_df = (selected_df
    # rename all columns to lowercase first
    .toDF(*[c.lower() for c in selected_df.columns])
)

selected_df = (selected_df
    .withColumn("flightdate", to_date(col("flightdate"), "yyyy-MM-dd"))
    .withColumn("reporting_airline", col("reporting_airline"))  # already string, no cast needed
    .withColumn("flight_number_reporting_airline", col("flight_number_reporting_airline").cast("int"))
    .withColumn("originairportid", col("originairportid").cast("int"))
    .withColumn("destairportid", col("destairportid").cast("int"))
    .withColumn("crsdeptime", col("crsdeptime").cast("int"))
    .withColumn("origin", col("origin"))
    .withColumn("dest", col("dest"))
    .withColumn("deptime", col("deptime").cast("int"))
    .withColumn("depdelay", col("depdelay").cast("double"))
    .withColumn("arrtime", col("arrtime").cast("int"))
    .withColumn("arrdelay", col("arrdelay").cast("double"))
    .withColumn("cancelled", col("cancelled").cast("double").cast("boolean"))
    .withColumn("cancellationcode", col("cancellationcode"))  # single-letter string, no cast needed
    .withColumn("diverted", col("diverted").cast("double").cast("boolean"))
    .withColumn("distance", col("distance").cast("double"))
    .withColumn("carrierdelay", col("carrierdelay").cast("double"))
    .withColumn("weatherdelay", col("weatherdelay").cast("double"))
    .withColumn("nasdelay", col("nasdelay").cast("double"))
    .withColumn("securitydelay", col("securitydelay").cast("double"))
    .withColumn("lateaircraftdelay", col("lateaircraftdelay").cast("double"))
)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 23, Finished, Available, Finished, False)

In [22]:
selected_df.printSchema()
display(null_percentage(selected_df))

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 24, Finished, Available, Finished, False)

root
 |-- flightdate: date (nullable = true)
 |-- reporting_airline: string (nullable = true)
 |-- flight_number_reporting_airline: integer (nullable = true)
 |-- originairportid: integer (nullable = true)
 |-- destairportid: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- crsdeptime: integer (nullable = true)
 |-- deptime: integer (nullable = true)
 |-- depdelay: double (nullable = true)
 |-- arrtime: integer (nullable = true)
 |-- arrdelay: double (nullable = true)
 |-- cancelled: boolean (nullable = true)
 |-- cancellationcode: string (nullable = true)
 |-- diverted: boolean (nullable = true)
 |-- distance: double (nullable = true)
 |-- carrierdelay: double (nullable = true)
 |-- weatherdelay: double (nullable = true)
 |-- nasdelay: double (nullable = true)
 |-- securitydelay: double (nullable = true)
 |-- lateaircraftdelay: double (nullable = true)



SynapseWidget(Synapse.DataFrame, b2622838-e95e-48ac-aafc-9a880c09d4d4)

In [23]:
from pyspark.sql.functions import col, lpad, concat_ws, substring, when

def format_hhmm(colname):
    padded = lpad(col(colname).cast("string"), 4, "0")
    return concat_ws(":", substring(padded, 1, 2), substring(padded, 3, 2))

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 25, Finished, Available, Finished, False)

In [24]:
selected_df = (selected_df
    .withColumn("crsdeptime_fmt", when(col("crsdeptime").isNotNull(), format_hhmm("crsdeptime")))
    .withColumn("deptime_fmt", when(col("deptime").isNotNull(), format_hhmm("deptime")))
    .withColumn("arrtime_fmt", when(col("arrtime").isNotNull(), format_hhmm("arrtime")))
)

display(selected_df)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a444aac0-948c-4ecf-a6bd-ac71603d8e01)

In [25]:
compare = selected_df.select("crsdeptime", "crsdeptime_fmt")
display(compare)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d886923a-892d-401a-9fea-1999f72ead80)

In [26]:
selected_df = (selected_df
    .withColumn("crsdeptime", when(col("crsdeptime").isNotNull(), format_hhmm("crsdeptime")))
    .withColumn("deptime", when(col("deptime").isNotNull(), format_hhmm("deptime")))
    .withColumn("arrtime", when(col("arrtime").isNotNull(), format_hhmm("arrtime")))
)

display(selected_df)

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, daa888ea-b990-468c-838f-28195c7329d4)

In [28]:
selected_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("silver_flights")

StatementMeta(, 8ef61a10-e4e8-4630-821b-9f1db1d020a1, 30, Finished, Available, Finished, False)

# Data Modelling

In [10]:
from pyspark.sql.functions import monotonically_increasing_id, col

dim_carrier = (spark.table("silver_flights")
    .select("reporting_airline")
    .distinct()
    .withColumn("carrier_key", monotonically_increasing_id())
    .select("carrier_key", col("reporting_airline").alias("carrier_code"))
)

dim_carrier.write.format("delta").mode("overwrite").saveAsTable("dim_carrier")

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 12, Finished, Available, Finished, False)

In [11]:
from pyspark.sql import functions as F

bounds = spark.table("silver_flights").select(
    F.min("flightdate").alias("min_d"),
    F.max("flightdate").alias("max_d")
)

dim_date = (
    bounds.select(
        F.explode(F.sequence(F.col("min_d"), F.col("max_d"), F.expr("interval 1 day"))).alias("date_key")
    )
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("day", F.dayofmonth("date_key"))
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("is_weekend", F.col("day_name").isin("Saturday", "Sunday"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 13, Finished, Available, Finished, False)

In [21]:
from pyspark.sql.functions import col, monotonically_increasing_id


origin_ids = spark.table("silver_flights").select(col("originairportid").alias("airport_id"))
dest_ids = spark.table("silver_flights").select(col("destairportid").alias("airport_id"))

dim_airport = (origin_ids.union(dest_ids)
    .distinct()
    .withColumn("airport_key", monotonically_increasing_id())
)


StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 23, Finished, Available, Finished, False)

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 24, Finished, Available, Finished, False)

root
 |-- airport_id: integer (nullable = true)
 |-- airport_key: long (nullable = false)



In [23]:
origin_pairs = (spark.table("silver_flights")
    .select(col("originairportid").alias("airport_id"), col("origin").alias("iata_code"))
)
dest_pairs = (spark.table("silver_flights")
    .select(col("destairportid").alias("airport_id"), col("dest").alias("iata_code"))
)

dim_airport = (origin_pairs.union(dest_pairs)
    .distinct()
    .withColumn("airport_key", monotonically_increasing_id())
)

dim_airport.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_airport")

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 25, Finished, Available, Finished, False)

In [24]:
dim_airport.printSchema()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 26, Finished, Available, Finished, False)

root
 |-- airport_id: integer (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airport_key: long (nullable = false)



In [25]:
from pyspark.sql.functions import split, col, when

silver = spark.table("silver_flights")

silver = silver.withColumn("crs_hour", split(col("crsdeptime"), ":").getItem(0).cast("int"))
silver = silver.withColumn("time_of_day",
    when((col("crs_hour") >= 5) & (col("crs_hour") < 12), "Morning")
    .when((col("crs_hour") >= 12) & (col("crs_hour") < 17), "Afternoon")
    .when((col("crs_hour") >= 17) & (col("crs_hour") < 21), "Evening")
    .otherwise("Night")
)


dim_carrier = spark.table("dim_carrier")
dim_airport = spark.table("dim_airport")

dim_date = spark.table("dim_date")

fact_flights = (silver
    .join(dim_carrier, silver.reporting_airline == dim_carrier.carrier_code, "left")
    .join(dim_airport.alias("origin_dim"), silver.originairportid == col("origin_dim.airport_id"), "left")
    .join(dim_airport.alias("dest_dim"), silver.destairportid == col("dest_dim.airport_id"), "left")
    .join(dim_date, silver.flightdate == dim_date.date_key, "left")
    .select(
        col("date_key"),
        "carrier_key",
        col("origin_dim.airport_key").alias("origin_airport_key"),
        col("dest_dim.airport_key").alias("dest_airport_key"),
        "time_of_day",
        "depdelay", "arrdelay", "distance",
        "cancelled", "diverted", "cancellationcode",
        "carrierdelay", "weatherdelay", "nasdelay", "securitydelay", "lateaircraftdelay"
    )
)

fact_flights.write.format("delta").mode("overwrite").saveAsTable("fact_flights")

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 27, Finished, Available, Finished, False)

# Enriching using OpenFlights

In [15]:
openflights_cols = ["airport_id_of", "name", "city", "country", "iata", "icao",
                     "latitude", "longitude", "altitude", "timezone", "dst",
                     "tz_database", "type", "source"]

openflights = spark.read.csv(
    "Files/bronze/raw_extracted/airports.dat",  
    header=False,
    inferSchema=True
).toDF(*openflights_cols)

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 17, Finished, Available, Finished, False)

In [26]:
openflights.printSchema()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 28, Finished, Available, Finished, False)

root
 |-- airport_id_of: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- timezone: string (nullable = true)
 |-- dst: string (nullable = true)
 |-- tz_database: string (nullable = true)
 |-- type: string (nullable = true)
 |-- source: string (nullable = true)



In [27]:
origin_pairs = spark.table("silver_flights").select(col("originairportid").alias("airport_id"), col("origin").alias("iata_code"))
dest_pairs = spark.table("silver_flights").select(col("destairportid").alias("airport_id"), col("dest").alias("iata_code"))

dim_airport_plain = (origin_pairs.union(dest_pairs)
    .distinct()
    .withColumn("airport_key", monotonically_increasing_id())
)

# Now enrich ONCE, aliasing to avoid any collision
dim_airport_enriched = (dim_airport_plain
    .join(
        openflights.select(
            col("iata").alias("of_iata"),
            "name", "city", "country",
            col("latitude").alias("of_latitude"),
            col("longitude").alias("of_longitude")
        ),
        dim_airport_plain.iata_code == col("of_iata"),
        "left"
    )
    .drop("of_iata")
    .withColumnRenamed("of_latitude", "latitude")
    .withColumnRenamed("of_longitude", "longitude")
)

dim_airport_enriched.printSchema()

# The core lesson here: once you save a table, don't read it back in and re-run the same enrichment join against it — that's what's compounding the duplication every time you rerun this cell.


StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 29, Finished, Available, Finished, False)

root
 |-- airport_id: integer (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airport_key: long (nullable = false)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [28]:
from pyspark.sql.functions import col, count

dim_airport_enriched.filter(col("latitude").isNull() | (col("longitude").isNull())).count()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 30, Finished, Available, Finished, False)

1

In [29]:
dim_airport_enriched = dim_airport_enriched.dropna()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 31, Finished, Available, Finished, False)

In [30]:
dim_airport_enriched.filter(col("latitude").isNull() | (col("longitude").isNull())).show()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 32, Finished, Available, Finished, False)

+----------+---------+-----------+----+----+-------+--------+---------+
|airport_id|iata_code|airport_key|name|city|country|latitude|longitude|
+----------+---------+-----------+----+----+-------+--------+---------+
+----------+---------+-----------+----+----+-------+--------+---------+



In [31]:
dim_airport_enriched.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_airport")

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 33, Finished, Available, Finished, False)

In [33]:
airport_dates = (spark.table("fact_flights")
    .select("origin_airport_key", "date_key")
    .distinct()
    .join(spark.table("dim_airport"), col("origin_airport_key") == col("airport_key"))
    .select("iata_code", "latitude", "longitude", "date_key")
)

airport_dates.count()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 35, Finished, Available, Finished, False)

9999

In [34]:
airport_dates.select("iata_code").distinct().count()

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 36, Finished, Available, Finished, False)

333

In [35]:
from pyspark.sql.functions import min as spark_min, max as spark_max

airports_to_query = (airport_dates
    .groupBy("iata_code", "latitude", "longitude")
    .agg(
        spark_min("date_key").alias("min_date"),
        spark_max("date_key").alias("max_date")
    )
    .collect()
)

display(airports_to_query)

StatementMeta(, 49b6b727-8331-43ef-96a6-d27659c73279, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 580d15a9-1de5-425b-b08a-b728069e8e18)

In [2]:
from pyspark.sql.functions import to_date, col

weather_raw = spark.read.csv(
    "Files/bronze/raw_extracted/weather_results.csv",
    header=True,
    inferSchema=False  
)

fact_weather = (weather_raw
    .withColumn("date", to_date(col("date")))
    .withColumn("precipitation", col("precipitation").cast("double"))
    .withColumn("windspeed", col("windspeed").cast("double"))
    .withColumn("temp_max", col("temp_max").cast("double"))
    .withColumn("temp_min", col("temp_min").cast("double"))
)

fact_weather.printSchema()
display(fact_weather.limit(10))

StatementMeta(, 97f2212e-a89c-453f-8ee2-ace2d82fac4b, 4, Finished, Available, Finished, False)

root
 |-- iata_code: string (nullable = true)
 |-- date: date (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)



SynapseWidget(Synapse.DataFrame, cebbbfaf-9af5-4a29-8894-9275334ce21f)

In [3]:
fact_weather.select("iata_code").distinct().count()

StatementMeta(, 97f2212e-a89c-453f-8ee2-ace2d82fac4b, 5, Finished, Available, Finished, False)

332

In [4]:
fact_weather.write.format("delta").mode("overwrite").saveAsTable("fact_weather")

StatementMeta(, 97f2212e-a89c-453f-8ee2-ace2d82fac4b, 6, Finished, Available, Finished, False)

In [5]:
dim_airport = spark.table("dim_airport")
dim_date = spark.table("dim_date")

fact_weather = (fact_weather
    .join(dim_airport, fact_weather.iata_code == dim_airport.iata_code, "left")
    .join(dim_date, fact_weather.date == dim_date.date_key, "left")
    .select(
        col("airport_key"),
        col("date_key"),
        "precipitation",
        "windspeed",
        "temp_max",
        "temp_min"
    )
)

fact_weather.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("fact_weather")

StatementMeta(, 97f2212e-a89c-453f-8ee2-ace2d82fac4b, 7, Finished, Available, Finished, False)